# backfill_raw_re_gift_customfields
Resilient backfill for gift custom fields. Processes one category at a time,
upserts page-by-page, and checkpoints after every page so any timeout loses
at most one in-flight page.

**Run this notebook repeatedly until all 24 categories show `complete`.**

Requires `renxt_gift_cf_backfill_state` registered as Input + Output dataset.

In [ ]:
%run user_configuration.ipynb
%run renxt_core.ipynb


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
OUTPUT_DATASET     = "raw_re_gift_customfields"   # replace with UUID if available
CHECKPOINT_DATASET = "renxt_gift_cf_backfill_state"
GIFT_CF_URL        = f"{API_BASE}/gift/v1/gifts/customfields"

CATEGORIES = [
    'RG Acquisition Appeal',
    'RG Annual Receipt Number',
    'RG Billable',
    'RG Cancellation Method',
    'RG Cancellation Reason',
    'RG Payment Attempt',
    'RG Rejection Reason',
    'RG Save Attempt',
    'RG SignUp Age',
    'RG Signup Amount',
    'RG SignUp Date',
    'RG SignUp Location',
    'RG SignUp Name',
    'RG Signup Supplier',
    'RG SignUp Type',
    'RG Survey',
    'RG Survey Commitment',
    'RG Survey Longevity Confidence',
    'RG Upgrade Date',
    'RG Upgrade Last',
    'RG Upgrade Previous',
    'RG Upgrade Previous 2 Previous',
    'RG Verification Call',
    'Gift Acquisition Source',
]
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
# ── Checkpoint helpers ────────────────────────────────────────────────────────

def _read_checkpoint() -> pd.DataFrame:
    """Read the full backfill checkpoint table."""
    try:
        df = domo.read_dataframe(CHECKPOINT_DATASET, query="SELECT * FROM table")
        return df
    except Exception:
        return pd.DataFrame(columns=[
            "category", "status", "next_link", "rows_written", "updated_at"
        ])


def _get_category_state(checkpoint_df: pd.DataFrame, category: str) -> dict:
    """Return state dict for a category, or defaults if not seen before."""
    if checkpoint_df.empty or "category" not in checkpoint_df.columns:
        return {"status": "pending", "next_link": None, "rows_written": 0}
    row = checkpoint_df[checkpoint_df["category"] == category]
    if row.empty:
        return {"status": "pending", "next_link": None, "rows_written": 0}
    r = row.iloc[0]
    return {
        "status":       str(r.get("status", "pending")),
        "next_link":    str(r["next_link"]) if pd.notna(r.get("next_link")) and str(r.get("next_link","")).strip() not in ("","nan","None") else None,
        "rows_written": int(r.get("rows_written", 0) or 0),
    }


def _save_category_state(checkpoint_df: pd.DataFrame, category: str,
                          status: str, next_link, rows_written: int) -> pd.DataFrame:
    """Upsert one category row in the checkpoint table and write back to Domo."""
    now = datetime.now(timezone.utc).isoformat()
    new_row = pd.DataFrame([{
        "category":     category,
        "status":       status,
        "next_link":    next_link if next_link else "",
        "rows_written": rows_written,
        "updated_at":   now,
    }])

    if checkpoint_df.empty or "category" not in checkpoint_df.columns:
        updated = new_row
    else:
        other = checkpoint_df[checkpoint_df["category"] != category].copy()
        updated = pd.concat([other, new_row], ignore_index=True)

    domo.write_dataframe(
        domo_safe_cast(updated),
        dataset=CHECKPOINT_DATASET,
        update_method="REPLACE",
    )
    return updated


print("✅ checkpoint helpers loaded")


In [ ]:
# ── Per-page fetch and upsert ─────────────────────────────────────────────────

def _fetch_and_upsert_category(category: str, checkpoint_df: pd.DataFrame) -> pd.DataFrame:
    """
    Fetch all pages for one category, upserting each page immediately to Domo.
    Resumes from saved next_link if a previous run was cut off mid-category.
    Updates checkpoint after every page so progress is never lost.

    Returns updated checkpoint_df.
    """
    state = _get_category_state(checkpoint_df, category)

    if state["status"] == "complete":
        print(f"  ⏭  [{category}] already complete ({state['rows_written']:,} rows) — skipping")
        return checkpoint_df

    token_mgr    = TokenManager(interactive=False)
    sess         = requests.Session()
    rows_written = state["rows_written"]
    page         = 0

    # Resume from saved next_link, or start fresh
    if state["next_link"]:
        url    = state["next_link"]
        params = None
        print(f"  ↩️  [{category}] resuming from checkpoint (already wrote {rows_written:,} rows)")
    else:
        url    = GIFT_CF_URL
        params = {"category": category, "include_inactive": "true"}
        print(f"  🔄 [{category}] starting from beginning")

    while True:
        page += 1
        resp = api_request_with_auth("GET", url, token_mgr=token_mgr,
                                     params=params, session=sess)
        _raise_for_status_with_body(resp, context=f"[gift_customfields/{category}] page={page}")

        payload          = resp.json() if resp.text else {}
        items, next_link = extract_items_and_next(payload)

        if items:
            df_page = pd.json_normalize(items, sep=".")
            df_page["category"]        = category
            df_page["pulled_at_utc"]   = datetime.now(timezone.utc).isoformat()
            df_page["_endpoint"]       = "gift_customfields"
            df_page                    = domo_safe_cast(df_page)

            domo.write_dataframe(
                df_page,
                dataset=OUTPUT_DATASET,
                update_method="upsert",
                update_key="id",
            )
            rows_written += len(df_page)
            print(f"    page {page}: upserted {len(df_page):,} rows "
                  f"(total {rows_written:,}) | next_link: {'yes' if next_link else 'none'}")

        # Save checkpoint after every page — captures next_link for resume
        status       = "complete" if not next_link else "in_progress"
        checkpoint_df = _save_category_state(
            checkpoint_df, category, status,
            next_link=next_link, rows_written=rows_written,
        )

        if not next_link:
            print(f"  ✅ [{category}] complete — {rows_written:,} total rows")
            break

        # Follow next_link — params already encoded in the URL
        url    = next_link
        params = None

    return checkpoint_df


print("✅ fetch-and-upsert helper loaded")


In [ ]:
# ── Main backfill loop ────────────────────────────────────────────────────────
# Reads checkpoint, skips complete categories, processes the next pending/
# in_progress one. Run this notebook repeatedly until all categories complete.

checkpoint_df = _read_checkpoint()

# Show current state
print("Current backfill state:")
print("-" * 55)
total_complete = 0
next_to_run    = None

for cat in CATEGORIES:
    state = _get_category_state(checkpoint_df, cat)
    status = state["status"]
    rows   = state["rows_written"]
    marker = "✅" if status == "complete" else ("▶️ " if status == "in_progress" else "⏳")
    print(f"  {marker}  {cat:<40} {status:<12} {rows:>8,} rows")
    if status == "complete":
        total_complete += 1
    elif next_to_run is None:
        next_to_run = cat

print("-" * 55)
print(f"  {total_complete}/{len(CATEGORIES)} categories complete")
print()

if next_to_run is None:
    print("🎉 All categories complete! Backfill is done.")
    print("   You can now schedule schedule_raw_re_gift_customfields.ipynb for daily incremental runs.")
else:
    print(f"▶️  Processing next category: [{next_to_run}]")
    print()
    checkpoint_df = _fetch_and_upsert_category(next_to_run, checkpoint_df)
    print()
    complete_now = sum(
        1 for cat in CATEGORIES
        if _get_category_state(checkpoint_df, cat)["status"] == "complete"
    )
    print(f"Progress: {complete_now}/{len(CATEGORIES)} categories complete")
    if complete_now == len(CATEGORIES):
        print("🎉 All categories complete!")
